In [192]:
import matplotlib as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [193]:
columns_headers = [
    "time", "gender", "highschool_%", "study_year","faculty","unv_grade_2023","accomodation","allowance","scholarship","study_hours",
    "party_week","drinks_night","missed_classes", "failed_classes", "in_relationship","parent_approve","relation_parent"
]
df = pd.read_csv("Stats_survey.csv", names=columns_headers, header=0)

#### Domain Expert Message:

This is the raw data file, straight from the survey collected. You will see some responses on "what year were you in last year" have been left blank which indicates that the students previous year of study was not in university but rather high school. Since the point of the model is to be based on university students, these observations that have been left blank will be removed so that only students who were in university in 2023 and onwards are taken into consideration.

In [194]:
df.study_year

0      2nd Year
1      2nd Year
2      1st Year
3      2nd Year
4      2nd Year
         ...   
401         NaN
402    2nd Year
403    1st Year
404         NaN
405    1st Year
Name: study_year, Length: 406, dtype: str

In [195]:
df.study_year.isnull().sum()

np.int64(73)

In [ ]:
df.dropna(subset=['study_year'], axis=0, inplace=True)

Dropping time column as it doesnot play any important part in the data and dropping duplicates as it affects the data quality of overall dataset.

In [197]:
df.drop(columns = "time", inplace =True)

In [198]:
df.drop_duplicates(inplace = True)

In [199]:
df.shape

(332, 16)

In [200]:
df.isna().sum()

gender              0
highschool_%        3
study_year          0
faculty             4
unv_grade_2023     14
accomodation        3
allowance          15
scholarship         0
study_hours         0
party_week          0
drinks_night        0
missed_classes      1
failed_classes      0
in_relationship     1
parent_approve      2
relation_parent     1
dtype: int64

As per domain expert advice dropping unv_grade_2023 nulls as it is the target value and filling it arbitarily may change the dataset's interpretation.

In [201]:
df.dropna(subset= "unv_grade_2023",inplace = True)

In [202]:
df.isna().sum()

gender              0
highschool_%        2
study_year          0
faculty             3
unv_grade_2023      0
accomodation        1
allowance          14
scholarship         0
study_hours         0
party_week          0
drinks_night        0
missed_classes      1
failed_classes      0
in_relationship     1
parent_approve      1
relation_parent     1
dtype: int64

* Filling categorical column with low null values with mode and filling numerical column with low null values with mean.
* Filling categorical column with high null values with "Unknown" value.

In [203]:
df.fillna({"highschool_%" : df["highschool_%"].mean()},inplace= True)
df.fillna({"faculty":df["faculty"].mode()[0]},inplace= True)
df.fillna({"accomodation": df["accomodation"].mode()[0]},inplace =True)
df.fillna({"missed_classes": df["missed_classes"].mode()[0]},inplace =True)
df.fillna({"in_relationship": df["in_relationship"].mode()[0]},inplace =True)
df.fillna({"parent_approve": df["parent_approve"].mode()[0]},inplace =True)
df.fillna({"relation_parent": df["relation_parent"].mode()[0]},inplace =True)
df.fillna({"allowance": "unknown"},inplace =True)
df.isna().sum()

gender             0
highschool_%       0
study_year         0
faculty            0
unv_grade_2023     0
accomodation       0
allowance          0
scholarship        0
study_hours        0
party_week         0
drinks_night       0
missed_classes     0
failed_classes     0
in_relationship    0
parent_approve     0
relation_parent    0
dtype: int64

No null values or duplicate values in the data. Now we can perform further data preprocessing for machine learning.

#### Outlier Handling

In [204]:
def handle_outlier(series : pd.Series) -> pd.Series:
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)

    IQR = Q3 -Q1
    min_value = Q1 -(1.5 * IQR)
    max_value = Q3 +(1.5 * IQR)

    outliers = series[(series > max_value)| (series < min_value)]
    print(outliers)

    return df.drop(outliers.index ,axis = 0, inplace = True)

In [205]:
df["highschool_%"].aggregate(handle_outlier)

38     99.00
54     98.00
73     98.33
268    60.00
335    55.00
358    60.00
Name: highschool_%, dtype: float64


In [206]:
df.head()

,gender,highschool_%,study_year,faculty,unv_grade_2023,accomodation,allowance,scholarship,study_hours,party_week,drinks_night,missed_classes,failed_classes,in_relationship,parent_approve,relation_parent
0,Female,76.0,2nd Year,Arts & Social Sciences,72.0,Private accommodation/ stay with family/friends,R 4001- R 5000,No,8+,Only weekends,8+,3,0,Yes,Yes,Very close
1,Male,89.0,2nd Year,Economic & Management Sciences,75.0,Private accommodation/ stay with family/friends,R 7001 - R 8000,"Yes (NSFAS, etc...)",8+,Only weekends,3-5,4+,0,No,Yes,Very close
2,Male,76.0,1st Year,AgriSciences,55.0,Private accommodation/ stay with family/friends,R 4001- R 5000,No,3-5,2,8+,3,0,No,Yes,Very close
3,Male,89.0,2nd Year,Engineering,84.0,Private accommodation/ stay with family/friends,R 6001 - R 7000,No,3-5,3,8+,2,0,Yes,Yes,Very close
4,Female,74.0,2nd Year,Arts & Social Sciences,52.0,Private accommodation/ stay with family/friends,R 4001- R 5000,No,3-5,Only weekends,5-8,1,3,No,Yes,Fair


In [208]:
df.to_csv("student_survey_clean.csv", index = "False")